# Fully digital solo iOS idea scoring audit

Recomputes v4.0 B2B/Prosumer low/base/high scores, fatal-gate-aware desk decisions, and the research order for three fully digital iPhone app candidates. Inputs are evidence-bounded judgments, not forecasts.

In [1]:
from decimal import Decimal, ROUND_HALF_UP

WEIGHTS = {
    'problem_demand': 25,
    'monetization_economics': 25,
    'competitive_wedge': 15,
    'distribution_access': 15,
    'repeat_value_retention': 15,
    'timing': 5,
}

IDEAS = {
    'Voice Memo Archive': {
        'scores': {
            'problem_demand': (3, 4, 4), 'monetization_economics': (2, 3, 3),
            'competitive_wedge': (3, 3, 4), 'distribution_access': (3, 4, 4),
            'repeat_value_retention': (2, 2, 3), 'timing': (3, 4, 4),
        },
        'evidence': 'C', 'gates': ['clear', 'unknown', 'clear', 'clear'],
        'test_cost': 1, 'solo_ops': 'low after feasibility pass',
    },
    'Vertical Video Preflight': {
        'scores': {
            'problem_demand': (3, 3, 4), 'monetization_economics': (2, 3, 3),
            'competitive_wedge': (2, 3, 3), 'distribution_access': (3, 3, 4),
            'repeat_value_retention': (3, 4, 4), 'timing': (3, 3, 4),
        },
        'evidence': 'C', 'gates': ['clear'] * 4,
        'test_cost': 2, 'solo_ops': 'medium',
    },
    'Shortcuts Payload Inspector': {
        'scores': {
            'problem_demand': (2, 3, 3), 'monetization_economics': (2, 3, 3),
            'competitive_wedge': (2, 3, 3), 'distribution_access': (3, 3, 4),
            'repeat_value_retention': (2, 3, 3), 'timing': (2, 3, 3),
        },
        'evidence': 'C', 'gates': ['clear'] * 4,
        'test_cost': 1, 'solo_ops': 'low',
    },
}


In [2]:
def round_half_up(value):
    return int(Decimal(str(value)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

def opportunity_interval(spec):
    return tuple(
        round_half_up(sum(WEIGHTS[d] * spec['scores'][d][i] / 5 for d in WEIGHTS))
        for i in range(3)
    )

def classification(interval):
    low, _, high = interval
    if high < 40:
        return 'weak'
    if low >= 60:
        return 'attractive'
    return 'borderline'

def desk_decision(interval, evidence, gates):
    low, _, high = interval
    if 'blocked' in gates:
        return 'ELENDIR'
    if 'unknown' in gates or evidence == 'D':
        return 'BELIRSIZ'
    if high < 40:
        return 'ELENDIR'
    if low >= 60:
        return 'TEST_ONCELIKLI'
    return 'TESTE_DEGER'


In [3]:
expected = {
    'Voice Memo Archive': ((52, 66, 72), 'BELIRSIZ'),
    'Vertical Video Preflight': ((52, 63, 72), 'TESTE_DEGER'),
    'Shortcuts Payload Inspector': ((43, 60, 63), 'TESTE_DEGER'),
}

rows = []
for name, spec in IDEAS.items():
    interval = opportunity_interval(spec)
    decision = desk_decision(interval, spec['evidence'], spec['gates'])
    assert (interval, decision) == expected[name], (name, interval, decision)
    assert classification(interval) == 'borderline'
    rows.append((name, *interval, classification(interval), spec['evidence'], decision, spec['solo_ops']))

headers = ('idea', 'low', 'base', 'high', 'class', 'evidence', 'decision', 'solo_ops')
print(' | '.join(headers))
print(' | '.join(['---'] * len(headers)))
for row in rows:
    print(' | '.join(map(str, row)))
print('\nPASS: arithmetic, classification, and fatal-gate-aware decisions')


idea | low | base | high | class | evidence | decision | solo_ops
--- | --- | --- | --- | --- | --- | --- | ---
Voice Memo Archive | 52 | 66 | 72 | borderline | C | BELIRSIZ | low after feasibility pass
Vertical Video Preflight | 52 | 63 | 72 | borderline | C | TESTE_DEGER | medium
Shortcuts Payload Inspector | 43 | 60 | 63 | borderline | C | TESTE_DEGER | low

PASS: arithmetic, classification, and fatal-gate-aware decisions


In [4]:
# Research sequence is not a product-success ranking. Clear gates come first for build candidacy,
# while a cheap technical RAT can still be the first learning action.
cleared = [name for name, spec in IDEAS.items() if 'unknown' not in spec['gates'] and 'blocked' not in spec['gates']]
cleared.sort(key=lambda name: (-opportunity_interval(IDEAS[name])[1], IDEAS[name]['test_cost']))
assert cleared == ['Vertical Video Preflight', 'Shortcuts Payload Inspector']
print('Cleared candidate order:', ' > '.join(cleared))
print('Cheapest uncertainty-reduction action: Voice Memo Archive metadata + 2 GB batch RAT')
print('PASS: founder-fit research sequence assertions')


Cleared candidate order: Vertical Video Preflight > Shortcuts Payload Inspector
Cheapest uncertainty-reduction action: Voice Memo Archive metadata + 2 GB batch RAT
PASS: founder-fit research sequence assertions
